# 🌸 FahMai RAG Challenge — Super AI Engineer Season 6

**Notebook นี้เป็นระบบ RAG (Retrieval-Augmented Generation) สำหรับการแข่งขัน FahMai RAG Challenge**

โดยใช้ Thai LLM ผ่าน API ของ thaillm.or.th (แต่ละ model มี endpoint แยกกัน)

### โครงสร้าง Notebook
1. **Setup & Configuration** — ติดตั้ง dependencies และตั้งค่า API endpoints ที่ถูกต้อง
2. **Load Knowledge Base** — โหลดไฟล์ markdown 118 ไฟล์
3. **Text Chunking & Indexing** — แบ่งข้อความและสร้าง index (TF-IDF + BM25)
4. **Retrieval Functions** — ฟังก์ชันค้นหาข้อมูลแบบ hybrid
5. **Prompt Engineering** — ออกแบบ prompt สำหรับ Thai LLM
6. **LLM Inference Functions** — เรียกใช้ LLM ด้วย requests library (apikey header)
7. **Main RAG Pipeline** — pipeline หลักพร้อม checkpoint/resume
8. **Multi-Model Ensemble** — ใช้หลาย model และ majority voting
9. **Generate Submission** — สร้างไฟล์ submission.csv
10. **Analysis & Debugging** — วิเคราะห์และ debug ผลลัพธ์

## Section 1: Setup & Configuration

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'rank_bm25', 'scikit-learn', 'pandas', 'tqdm'])


In [ ]:
import os
import re
import json
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from collections import Counter

# ============================================================
# API CONFIGURATION
# IMPORTANT: Each model has its own separate endpoint.
# Authentication uses 'apikey' header (NOT 'Authorization: Bearer').
# The model name in the request body is always "/model".
# ============================================================

API_KEY = "zEJ2JyfKmDAguvcUgVndXkNGF7g19vsf"

# Each model has its own base URL under http://thaillm.or.th
MODEL_ENDPOINTS = {
    "openthaigpt": "http://thaillm.or.th/api/openthaigpt/v1",
    "pathumma":    "http://thaillm.or.th/api/pathumma/v1",
    "typhoon":     "http://thaillm.or.th/api/typhoon/v1",
    "kbtg":        "http://thaillm.or.th/api/kbtg/v1",
}

# Human-readable model names (for display)
MODEL_NAMES = {
    "openthaigpt": "OpenThaiGPT-ThaiLLM-8B-instruct-v7.2",
    "pathumma":    "Pathumma-ThaiLLM-qwen3-8b-think-3.0.0",
    "typhoon":     "Typhoon-S-ThaiLLM-8B-Instruct",
    "kbtg":        "THaLLE-0.2-ThaiLLM-8b-fa",
}

# Primary model used for single-model inference
PRIMARY_MODEL = "openthaigpt"

# Request defaults
DEFAULT_MAX_TOKENS = 512
DEFAULT_TEMPERATURE = 0.1
REQUEST_TIMEOUT = 60  # seconds

# Paths
BASE_DIR = Path("data")
KB_DIR = BASE_DIR / "knowledge_base"
QUESTIONS_FILE = BASE_DIR / "questions.csv"
CHECKPOINT_FILE = Path("rag_checkpoint.json")
SUBMISSION_FILE = Path("submission.csv")

print("Configuration loaded.")
print(f"Primary model: {MODEL_NAMES[PRIMARY_MODEL]}")
print(f"Primary endpoint: {MODEL_ENDPOINTS[PRIMARY_MODEL]}")
print(f"Available models: {list(MODEL_ENDPOINTS.keys())}")

## Section 2: Load Knowledge Base

In [ ]:
def load_knowledge_base(kb_dir: Path) -> list[dict]:
    """
    Load all .md files from the knowledge base directory.
    Returns a list of dicts with keys: path, category, filename, content.
    """
    documents = []
    md_files = sorted(kb_dir.rglob("*.md"))

    for fp in md_files:
        try:
            content = fp.read_text(encoding="utf-8").strip()
        except Exception as e:
            print(f"[WARN] Cannot read {fp}: {e}")
            continue

        # Derive category from subfolder name
        try:
            category = fp.relative_to(kb_dir).parts[0] if len(fp.relative_to(kb_dir).parts) > 1 else "root"
        except ValueError:
            category = "root"

        documents.append({
            "path": str(fp),
            "category": category,
            "filename": fp.name,
            "content": content,
        })

    return documents


documents = load_knowledge_base(KB_DIR)
print(f"Loaded {len(documents)} documents from knowledge base.")

# Show category breakdown
cat_counts = Counter(d["category"] for d in documents)
for cat, cnt in sorted(cat_counts.items()):
    print(f"  {cat}: {cnt} files")

# Preview first document
if documents:
    print("\n--- Preview of first document ---")
    print(f"File: {documents[0]['filename']}")
    print(documents[0]['content'][:500])

In [ ]:
# Load the 100 questions
questions_df = pd.read_csv(QUESTIONS_FILE)
print(f"Loaded {len(questions_df)} questions.")
print(f"Columns: {list(questions_df.columns)}")
print("\nFirst 3 rows:")
display(questions_df.head(3))

## Section 3: Text Chunking & Indexing (TF-IDF + BM25)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi


def simple_thai_tokenize(text: str) -> list[str]:
    """
    Simple tokenizer: split on whitespace and punctuation.
    For better Thai word segmentation, replace with pythainlp if available.
    """
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"[^\u0E00-\u0E7Fa-zA-Z0-9\s]", " ", text)
    tokens = text.lower().split()
    return [t for t in tokens if len(t) >= 2]


def chunk_documents(documents: list[dict], chunk_size: int = 400, overlap: int = 50) -> list[dict]:
    """
    Split each document into chunks of approximately chunk_size characters,
    with some overlap to preserve context at boundaries.
    Returns a flat list of chunk dicts.
    """
    chunks = []
    for doc in documents:
        content = doc["content"]
        if len(content) <= chunk_size:
            chunks.append({
                **doc,
                "chunk_id": f"{doc['filename']}_0",
                "chunk_text": content,
            })
        else:
            start = 0
            chunk_idx = 0
            while start < len(content):
                end = min(start + chunk_size, len(content))
                chunk_text = content[start:end]
                chunks.append({
                    **doc,
                    "chunk_id": f"{doc['filename']}_{chunk_idx}",
                    "chunk_text": chunk_text,
                })
                chunk_idx += 1
                start += chunk_size - overlap
    return chunks


# Build chunks
chunks = chunk_documents(documents, chunk_size=500, overlap=50)
print(f"Total chunks: {len(chunks)}")

chunk_texts = [c["chunk_text"] for c in chunks]
tokenized_chunks = [simple_thai_tokenize(t) for t in chunk_texts]

# Build BM25 index
bm25_index = BM25Okapi(tokenized_chunks)
print("BM25 index built.")

# Build TF-IDF index
tfidf_vectorizer = TfidfVectorizer(
    tokenizer=simple_thai_tokenize,
    token_pattern=None,
    min_df=1,
    max_features=50000,
    sublinear_tf=True,
)
tfidf_matrix = tfidf_vectorizer.fit_transform(chunk_texts)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

## Section 4: Retrieval Functions (Keyword + TF-IDF + BM25 Hybrid)

In [ ]:
def retrieve_bm25(query: str, top_k: int = 5) -> list[dict]:
    """Retrieve top-k chunks using BM25 scoring."""
    tokens = simple_thai_tokenize(query)
    scores = bm25_index.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in top_indices:
        if scores[idx] > 0:
            results.append({**chunks[idx], "score": float(scores[idx]), "method": "bm25"})
    return results


def retrieve_tfidf(query: str, top_k: int = 5) -> list[dict]:
    """Retrieve top-k chunks using TF-IDF cosine similarity."""
    q_vec = tfidf_vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(sims)[::-1][:top_k]
    results = []
    for idx in top_indices:
        if sims[idx] > 0:
            results.append({**chunks[idx], "score": float(sims[idx]), "method": "tfidf"})
    return results


def retrieve_keyword(query: str, top_k: int = 5) -> list[dict]:
    """Retrieve chunks containing keywords from the query."""
    tokens = simple_thai_tokenize(query)
    # Use longer tokens as keywords (more specific)
    keywords = [t for t in tokens if len(t) >= 3]
    if not keywords:
        keywords = tokens

    results = []
    for chunk in chunks:
        text_lower = chunk["chunk_text"].lower()
        match_count = sum(1 for kw in keywords if kw in text_lower)
        if match_count > 0:
            results.append({**chunk, "score": match_count / len(keywords), "method": "keyword"})

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]


def hybrid_retrieve(query: str, top_k: int = 6) -> list[dict]:
    """
    Combine BM25 + TF-IDF + keyword retrieval.
    De-duplicate by chunk_id, merge scores with weighted sum.
    Returns the top_k most relevant chunks.
    """
    bm25_results   = retrieve_bm25(query, top_k=top_k)
    tfidf_results  = retrieve_tfidf(query, top_k=top_k)
    kw_results     = retrieve_keyword(query, top_k=top_k)

    # Merge: accumulate normalised scores per chunk_id
    scored: dict[str, dict] = {}

    def _add(results, weight):
        if not results:
            return
        max_score = max(r["score"] for r in results) or 1.0
        for r in results:
            cid = r["chunk_id"]
            norm = r["score"] / max_score
            if cid not in scored:
                scored[cid] = {**r, "hybrid_score": 0.0}
            scored[cid]["hybrid_score"] += weight * norm

    _add(bm25_results,  weight=0.5)
    _add(tfidf_results, weight=0.35)
    _add(kw_results,    weight=0.15)

    ranked = sorted(scored.values(), key=lambda x: x["hybrid_score"], reverse=True)
    return ranked[:top_k]


# Quick test
test_query = "ราคาสินค้า"
test_results = hybrid_retrieve(test_query, top_k=3)
print(f"Test query: '{test_query}'")
for r in test_results:
    print(f"  [{r['category']}] {r['filename']}  score={r['hybrid_score']:.3f}")
    print(f"  {r['chunk_text'][:120]}...")

## Section 5: Prompt Engineering (Thai Prompts, Few-shot Examples)

In [ ]:
SYSTEM_PROMPT = """คุณเป็น AI ผู้ช่วยที่เชี่ยวชาญในการตอบคำถามเกี่ยวกับร้านฝ้ายไหม (FahMai) โดยอ้างอิงจากข้อมูลในฐานข้อมูลที่ให้มาเท่านั้น

กฎสำคัญ:
1. อ่านข้อมูลที่ให้มาอย่างละเอียด
2. ตอบโดยอ้างอิงจากข้อมูลที่ให้มาเท่านั้น ห้ามใช้ความรู้ภายนอก
3. หากไม่มีข้อมูลในฐานข้อมูล ให้ตอบด้วยเลข 9
4. หากคำถามไม่เกี่ยวข้องกับฐานข้อมูลสินค้าร้านฝ้ายไหม ให้ตอบด้วยเลข 10
5. ตอบด้วยตัวเลขเพียงอย่างเดียว (1-10) ห้ามอธิบายเพิ่มเติม"""

FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": """ข้อมูลจากฐานข้อมูล:
ผ้าไหมมัดหมี่ ราคา 1,500 บาท/ผืน มีสีแดง สีน้ำเงิน สีเขียว

คำถาม: ผ้าไหมมัดหมี่มีสีอะไรบ้าง?
ตัวเลือก:
1. สีแดง สีน้ำเงิน
2. สีแดง สีน้ำเงิน สีเขียว
3. สีเหลือง สีส้ม
4. สีขาว สีดำ
5. สีม่วง สีชมพู
6. สีเทา สีน้ำตาล
7. สีแดง สีเขียว สีเหลือง
8. สีน้ำเงิน สีม่วง สีเขียว
9. ไม่มีข้อมูลนี้ในฐานข้อมูล
10. คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า

ตอบด้วยตัวเลขเพียงอย่างเดียว:"""
    },
    {
        "role": "assistant",
        "content": "2"
    },
    {
        "role": "user",
        "content": """ข้อมูลจากฐานข้อมูล:
นโยบายการคืนสินค้า: สามารถคืนสินค้าได้ภายใน 7 วัน หากสินค้ามีตำหนิจากโรงงาน

คำถาม: ราคาหุ้นของบริษัท Apple วันนี้เป็นเท่าไร?
ตัวเลือก:
1. 150 ดอลลาร์
2. 175 ดอลลาร์
3. 200 ดอลลาร์
4. 120 ดอลลาร์
5. 160 ดอลลาร์
6. 180 ดอลลาร์
7. 190 ดอลลาร์
8. 145 ดอลลาร์
9. ไม่มีข้อมูลนี้ในฐานข้อมูล
10. คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า

ตอบด้วยตัวเลขเพียงอย่างเดียว:"""
    },
    {
        "role": "assistant",
        "content": "10"
    },
]


def build_rag_prompt(question: str, choices: list[str], context_chunks: list[dict]) -> list[dict]:
    """
    Build the full message list for the RAG query.
    Inserts retrieved context + question + choices as the final user message.
    """
    # Build context string from retrieved chunks
    if context_chunks:
        context_parts = []
        for i, chunk in enumerate(context_chunks, 1):
            context_parts.append(f"[ข้อมูลที่ {i}] (ไฟล์: {chunk['filename']})\n{chunk['chunk_text']}")
        context_str = "\n\n".join(context_parts)
    else:
        context_str = "ไม่พบข้อมูลที่เกี่ยวข้องในฐานข้อมูล"

    # Build choices string
    choices_str = "\n".join(f"{i+1}. {c}" for i, c in enumerate(choices))

    user_message = f"""ข้อมูลจากฐานข้อมูล:
{context_str}

คำถาม: {question}
ตัวเลือก:
{choices_str}

ตอบด้วยตัวเลขเพียงอย่างเดียว (1-10):"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        *FEW_SHOT_EXAMPLES,
        {"role": "user", "content": user_message},
    ]
    return messages


print("Prompt engineering functions ready.")
print(f"System prompt length: {len(SYSTEM_PROMPT)} chars")
print(f"Few-shot examples: {len(FEW_SHOT_EXAMPLES)//2} pairs")

## Section 6: LLM Inference Functions

Using the `requests` library directly because the API uses a non-standard `apikey` header (not `Authorization: Bearer`).

In [ ]:
def strip_think_tags(text: str) -> str:
    """
    Remove <think>...</think> blocks that some models (e.g. Pathumma/qwen3-think)
    prepend before giving the actual answer.
    """
    # Remove <think> ... </think> blocks (greedy, handles multiline)
    text = re.sub(r"<think>[\s\S]*?</think>", "", text, flags=re.IGNORECASE)
    return text.strip()


def call_llm(
    messages: list[dict],
    model_key: str = PRIMARY_MODEL,
    max_tokens: int = DEFAULT_MAX_TOKENS,
    temperature: float = DEFAULT_TEMPERATURE,
    retries: int = 3,
    retry_delay: float = 5.0,
) -> str:
    """
    Call the Thai LLM API using the requests library.

    Authentication uses 'apikey' header (lowercase), NOT 'Authorization: Bearer'.
    The model name in the request body is always '/model'.
    Each model has its own base_url from MODEL_ENDPOINTS.

    Returns the raw response text (with <think> tags stripped).
    """
    base_url = MODEL_ENDPOINTS[model_key]
    url = f"{base_url}/chat/completions"

    headers = {
        "Content-Type": "application/json",
        "apikey": API_KEY,  # NOT 'Authorization: Bearer'
    }

    payload = {
        "model": "/model",  # Always '/model', regardless of which endpoint
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }

    for attempt in range(1, retries + 1):
        try:
            resp = requests.post(
                url,
                headers=headers,
                json=payload,
                timeout=REQUEST_TIMEOUT,
            )
            resp.raise_for_status()
            data = resp.json()
            raw_text = data["choices"][0]["message"]["content"]
            return strip_think_tags(raw_text)

        except requests.exceptions.Timeout:
            print(f"[WARN] Timeout on attempt {attempt}/{retries} for model {model_key}")
        except requests.exceptions.HTTPError as e:
            print(f"[WARN] HTTP error {e.response.status_code} on attempt {attempt}/{retries}: {e}")
        except Exception as e:
            print(f"[WARN] Unexpected error on attempt {attempt}/{retries}: {e}")

        if attempt < retries:
            time.sleep(retry_delay)

    return ""  # Return empty string if all retries failed


def parse_answer(raw_text: str) -> int:
    """
    Parse the model's response to extract a number between 1 and 10.
    Falls back to 9 ('no data in KB') if no valid number is found.
    """
    if not raw_text:
        return 9

    # Find the first standalone digit 1-10
    # Try '10' first to avoid matching just '1' from '10'
    match = re.search(r'\b(10|[1-9])\b', raw_text)
    if match:
        return int(match.group(1))

    # Fallback: first digit in string
    digits = re.findall(r'[1-9]|10', raw_text)
    if digits:
        return int(digits[0])

    return 9  # Default: no data in KB


print("LLM inference functions ready.")

# Test strip_think_tags
test_response = "<think>\nLet me think about this...\n</think>\n2"
print(f"strip_think_tags test: '{strip_think_tags(test_response)}'")

# Test parse_answer
print(f"parse_answer('2') = {parse_answer('2')}")
print(f"parse_answer('The answer is 10.') = {parse_answer('The answer is 10.')}")
print(f"parse_answer('') = {parse_answer('')}")

In [ ]:
# ============================================================
# Quick API connectivity test
# ============================================================

def test_api_connection(model_key: str = PRIMARY_MODEL) -> bool:
    """Send a simple greeting to verify the API is reachable."""
    print(f"Testing connection to model: {MODEL_NAMES[model_key]}")
    print(f"Endpoint: {MODEL_ENDPOINTS[model_key]}")
    try:
        response = call_llm(
            messages=[{"role": "user", "content": "สวัสดี ตอบสั้นๆ ด้วย"}],
            model_key=model_key,
            max_tokens=50,
            temperature=0.3,
            retries=2,
        )
        if response:
            print(f"SUCCESS — response: {response[:100]}")
            return True
        else:
            print("FAILED — empty response")
            return False
    except Exception as e:
        print(f"FAILED — {e}")
        return False


# Run test
test_api_connection(PRIMARY_MODEL)

## Section 7: Main RAG Pipeline (with Checkpoint/Resume)

In [ ]:
def get_question_columns(df: pd.DataFrame) -> tuple[str, list[str]]:
    """
    Auto-detect the question column and choice columns from the DataFrame.
    Returns (question_col, [choice_cols]).
    """
    cols = list(df.columns)

    # Detect question column
    question_col = None
    for candidate in ["question", "คำถาม", "Question", "q"]:
        if candidate in cols:
            question_col = candidate
            break
    if question_col is None:
        # Guess: first non-id string column
        for c in cols:
            if c.lower() not in ("id", "answer") and df[c].dtype == object:
                question_col = c
                break

    # Detect choice columns (choice_1..choice_10 or choice1..choice10)
    choice_cols = [c for c in cols if re.match(r'choice[_\s]?\d+', c, re.IGNORECASE)]
    choice_cols.sort(key=lambda x: int(re.search(r'\d+', x).group()))

    if not choice_cols:
        # Fallback: look for columns named 1-10 or option_1..10
        choice_cols = [c for c in cols if re.match(r'(option|opt|ans|choice)?[_\s]?\d+$', c, re.IGNORECASE) and c not in (question_col, 'id', 'answer')]
        choice_cols.sort(key=lambda x: int(re.search(r'\d+', x).group()))

    return question_col, choice_cols


question_col, choice_cols = get_question_columns(questions_df)
print(f"Question column: '{question_col}'")
print(f"Choice columns ({len(choice_cols)}): {choice_cols}")

In [ ]:
def load_checkpoint() -> dict:
    """Load saved checkpoint (partial results)."""
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_checkpoint(results: dict) -> None:
    """Persist current results to disk."""
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)


def run_rag_pipeline(
    df: pd.DataFrame,
    model_key: str = PRIMARY_MODEL,
    top_k: int = 6,
    sleep_between: float = 0.5,
    force_rerun: bool = False,
) -> dict:
    """
    Run the RAG pipeline over all questions in df.
    Supports checkpoint/resume: already-answered questions are skipped.

    Returns dict mapping question id -> answer (int 1-10).
    """
    results = {} if force_rerun else load_checkpoint()
    already_done = len(results)
    if already_done:
        print(f"Resuming from checkpoint: {already_done} questions already answered.")

    q_col, c_cols = get_question_columns(df)

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"RAG [{model_key}]"):
        qid = str(row["id"])

        if qid in results:
            continue  # Skip already processed

        question = str(row[q_col])
        choices  = [str(row[c]) for c in c_cols if c in row.index]

        # Retrieve relevant context
        context_chunks = hybrid_retrieve(question, top_k=top_k)

        # Build prompt
        messages = build_rag_prompt(question, choices, context_chunks)

        # Call LLM
        raw = call_llm(messages, model_key=model_key)

        # Parse answer
        answer = parse_answer(raw)

        results[qid] = {
            "answer": answer,
            "raw": raw,
            "model": model_key,
            "context_files": [c["filename"] for c in context_chunks],
        }

        save_checkpoint(results)

        if sleep_between > 0:
            time.sleep(sleep_between)

    print(f"\nDone. {len(results)}/{len(df)} questions answered.")
    return results


print("Pipeline functions ready.")

In [ ]:
# ============================================================
# Run the main RAG pipeline (single model)
# Set force_rerun=True to ignore checkpoint and start fresh
# ============================================================

single_model_results = run_rag_pipeline(
    df=questions_df,
    model_key=PRIMARY_MODEL,
    top_k=6,
    sleep_between=0.5,
    force_rerun=False,
)

# Quick summary
answer_counts = Counter(v["answer"] for v in single_model_results.values())
print("\nAnswer distribution:")
for ans in sorted(answer_counts):
    label = "(no data)" if ans == 9 else "(unrelated)" if ans == 10 else ""
    print(f"  Choice {ans}: {answer_counts[ans]} times {label}")

## Section 8: Multi-Model Ensemble

In [ ]:
def run_ensemble(
    df: pd.DataFrame,
    model_keys: list[str] = None,
    top_k: int = 6,
    sleep_between: float = 0.5,
) -> dict:
    """
    Run the RAG pipeline with multiple models and combine using majority voting.
    Each model's results are saved to a separate checkpoint file.

    Returns dict mapping question id -> final answer (int 1-10).
    """
    if model_keys is None:
        model_keys = list(MODEL_ENDPOINTS.keys())

    all_model_results = {}

    for model_key in model_keys:
        print(f"\n{'='*60}")
        print(f"Running model: {MODEL_NAMES[model_key]}")
        print(f"{'='*60}")

        # Use per-model checkpoint files
        global CHECKPOINT_FILE
        original_checkpoint = CHECKPOINT_FILE
        CHECKPOINT_FILE = Path(f"rag_checkpoint_{model_key}.json")

        model_results = run_rag_pipeline(
            df=df,
            model_key=model_key,
            top_k=top_k,
            sleep_between=sleep_between,
        )
        all_model_results[model_key] = model_results

        CHECKPOINT_FILE = original_checkpoint

    # Majority voting
    print("\nApplying majority voting...")
    final_answers = {}
    q_col, _ = get_question_columns(df)

    for _, row in df.iterrows():
        qid = str(row["id"])
        votes = []
        for model_key in model_keys:
            if qid in all_model_results.get(model_key, {}):
                votes.append(all_model_results[model_key][qid]["answer"])

        if votes:
            # Majority vote; tie-break with PRIMARY_MODEL answer
            vote_counter = Counter(votes)
            final_answers[qid] = vote_counter.most_common(1)[0][0]
        else:
            final_answers[qid] = 9  # Fallback

    print(f"Ensemble complete. {len(final_answers)} answers.")
    return final_answers, all_model_results


# ============================================================
# Uncomment to run the full 4-model ensemble:
# (This takes significantly longer — ~4x single model time)
# ============================================================
# ensemble_answers, all_model_results = run_ensemble(
#     df=questions_df,
#     model_keys=["openthaigpt", "pathumma", "typhoon", "kbtg"],
#     top_k=6,
#     sleep_between=0.5,
# )

print("Ensemble function ready.")
print("To run ensemble, uncomment the call above.")

## Section 9: Generate Submission

In [ ]:
def generate_submission(
    results: dict,
    df: pd.DataFrame,
    output_path: Path = SUBMISSION_FILE,
    use_ensemble: bool = False,
) -> pd.DataFrame:
    """
    Generate submission.csv from pipeline results.

    results: either the output of run_rag_pipeline (dict[qid -> {answer, raw, ...}])
             or the first return value of run_ensemble (dict[qid -> int]).
    use_ensemble: if True, results values are ints directly; if False, they are dicts.
    """
    rows = []
    for _, row in df.iterrows():
        qid = str(row["id"])
        if use_ensemble:
            answer = results.get(qid, 9)
        else:
            answer = results.get(qid, {}).get("answer", 9)
        rows.append({"id": row["id"], "answer": answer})

    submission_df = pd.DataFrame(rows)[["id", "answer"]]
    submission_df.to_csv(output_path, index=False)
    print(f"Submission saved to: {output_path}")
    print(f"Total rows: {len(submission_df)}")
    return submission_df


# Generate from single-model results
submission_df = generate_submission(
    results=single_model_results,
    df=questions_df,
    output_path=SUBMISSION_FILE,
    use_ensemble=False,
)

print("\nSubmission preview:")
display(submission_df.head(10))

print("\nAnswer distribution:")
display(submission_df["answer"].value_counts().sort_index())

In [ ]:
# Validate submission format
assert list(submission_df.columns) == ["id", "answer"], "Columns must be ['id', 'answer']"
assert len(submission_df) == len(questions_df), f"Expected {len(questions_df)} rows, got {len(submission_df)}"
assert submission_df["answer"].between(1, 10).all(), "All answers must be between 1 and 10"
assert not submission_df["id"].duplicated().any(), "Duplicate IDs found!"

print("Submission validation passed!")
print(f"  Rows: {len(submission_df)}")
print(f"  Answer range: {submission_df['answer'].min()} – {submission_df['answer'].max()}")

## Section 10: Analysis & Debugging

In [ ]:
def analyze_results(results: dict, df: pd.DataFrame) -> None:
    """
    Print an analysis of the pipeline results:
    - Answer distribution
    - Empty/failed responses
    - Most frequently retrieved documents
    """
    q_col, _ = get_question_columns(df)
    answered = [v for v in results.values() if isinstance(v, dict)]

    print(f"Total answered: {len(answered)}/{len(df)}")
    empty = [v for v in answered if not v.get("raw")]
    print(f"Empty/failed responses: {len(empty)}")

    answer_counts = Counter(v["answer"] for v in answered)
    print("\nAnswer distribution:")
    for i in range(1, 11):
        label = ""
        if i == 9:  label = " ← ไม่มีข้อมูลในฐานข้อมูล"
        if i == 10: label = " ← คำถามไม่เกี่ยวข้อง"
        print(f"  {i:2d}: {answer_counts.get(i, 0):3d} times{label}")

    # Most retrieved documents
    all_files = []
    for v in answered:
        all_files.extend(v.get("context_files", []))
    print("\nTop 10 most retrieved documents:")
    for fname, cnt in Counter(all_files).most_common(10):
        print(f"  {fname}: {cnt} times")


analyze_results(single_model_results, questions_df)

In [ ]:
def debug_question(qid: str, results: dict, df: pd.DataFrame, top_k: int = 6) -> None:
    """
    Show full details for a single question: context retrieved, prompt, raw LLM response, answer.
    """
    q_col, c_cols = get_question_columns(df)

    row = df[df["id"].astype(str) == str(qid)]
    if row.empty:
        print(f"Question id '{qid}' not found.")
        return

    row = row.iloc[0]
    question = str(row[q_col])
    choices  = [str(row[c]) for c in c_cols if c in row.index]

    print(f"{'='*70}")
    print(f"Question ID: {qid}")
    print(f"Question: {question}")
    print("\nChoices:")
    for i, c in enumerate(choices, 1):
        print(f"  {i}. {c}")

    # Retrieve context
    context = hybrid_retrieve(question, top_k=top_k)
    print(f"\nRetrieved {len(context)} context chunks:")
    for i, chunk in enumerate(context, 1):
        print(f"\n  [{i}] {chunk['filename']} (score={chunk['hybrid_score']:.3f})")
        print(f"  {chunk['chunk_text'][:200]}...")

    # Show stored result
    stored = results.get(str(qid))
    if stored:
        print(f"\nModel raw response: {stored.get('raw', '(empty)')}")
        print(f"Parsed answer: {stored.get('answer')}")
    else:
        print("\nNo stored result for this question.")


# Debug example: change the qid to inspect any question
if questions_df is not None and len(questions_df) > 0:
    sample_qid = str(questions_df.iloc[0]["id"])
    debug_question(sample_qid, single_model_results, questions_df)

In [ ]:
# ============================================================
# Test API connectivity for all 4 models
# ============================================================

def test_all_models() -> None:
    """Run a quick connectivity check for every model endpoint."""
    for model_key in MODEL_ENDPOINTS:
        print(f"\nTesting {model_key} ({MODEL_NAMES[model_key]})...")
        success = test_api_connection(model_key)
        status = "OK" if success else "FAILED"
        print(f"  Status: {status}")


# Uncomment to test all models:
# test_all_models()

print("Run test_all_models() to check all 4 endpoints.")

In [ ]:
# ============================================================
# Manual one-off query — for interactive exploration
# ============================================================

def ask(
    question: str,
    model_key: str = PRIMARY_MODEL,
    top_k: int = 6,
    show_context: bool = True,
) -> None:
    """Ask a free-form question using the RAG pipeline."""
    context = hybrid_retrieve(question, top_k=top_k)

    if show_context:
        print(f"Retrieved {len(context)} context chunks:")
        for i, c in enumerate(context, 1):
            print(f"  [{i}] {c['filename']} (score={c['hybrid_score']:.3f})")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"""
ข้อมูลจากฐานข้อมูล:
{'\\n\\n'.join(c['chunk_text'] for c in context)}

คำถาม: {question}
"""},
    ]
    raw = call_llm(messages, model_key=model_key, max_tokens=512, temperature=0.3)
    print(f"\nResponse: {raw}")


# Example usage:
# ask("ผ้าไหมมัดหมี่ราคาเท่าไร?")
# ask("นโยบายการคืนสินค้าของร้านเป็นอย่างไร?")
print("ask() function ready for interactive use.")